In [23]:
from collections import Counter
import json
import os
import pandas as pd
import re

In [ ]:
class args:
    data_dir = "../saved_data/inference/"
    
    
    selected_models = ['llama3_3_70b', 'gpt_oss_120b', 'gpt_oss_20b', 'qwen_3_8B', 'llama4_17b_16e', 'qwen_3_30b_a3b_thinking', 'gemma_4_31B_it', 'claude_sonnet_4_5']

    
    
    # static_analyzer_data_path = "../data/static_analyzer_Downloaded_Data/"
    setting = ["batching", "single_variable"][1]
    saving_path = f"../saved_data/inference/{setting}"
    variable_column = "VarName"
    
    saving_path = "../saved_data/processed_inference/"


    experimental_setting = {
        "single_variable": ["full_source_code", "paragraph_code", "variable_k_lines", "app_name_only"][3], 
        "batching": ["paragraph_code_k_hop", "paragraph_code_record", "variable_k_lines_k_hop", "variable_k_lines_record"][0]
    }[setting]
    
    data_path = {
        "single_variable": '../data/get_variable_para_info_formatted_info_annotation_set_23_06_2026_with_path.json',
        "batching": '../data/batched_data/record/batched_data_record_25June2026.json'
        
    }[setting]

    include_voting = True

    # batching_context = [True, False][0]




In [31]:
def extract_batch_size(filename):
    # Searches for 'batch_size_' followed by one or more digits
    match = re.search(r'batch_size_(\+?\d+)', filename)
    return int(match.group(1)) if match else None


def batch_dataframe(df, batch_size):
    """
    Transforms each column into a series of lists of length batch_size.
    """
    batched_data = {}
    
    for col in df.columns:
        # Slice the column into chunks of batch_size
        column_values = df[col].tolist()
        batched_data[col] = [
            column_values[i : i + batch_size] 
            for i in range(0, len(column_values), batch_size)
        ]
        
    return pd.DataFrame(batched_data)

In [32]:
args.selected_models

['llama3_3_70b',
 'gpt_oss_120b',
 'gpt_oss_20b',
 'qwen_3_8B',
 'llama4_17b_16e',
 'qwen_3_30b_a3b_thinking',
 'gemma_4_31B_it',
 'claude_sonnet_4_5']

In [34]:
pred_dir = args.data_dir+args.setting+"/"+args.experimental_setting
pred_dir

'../saved_data/inference/single_variable/app_name_only'

In [35]:
# all_files = [x for x in os.listdir(pred_dir) if x.endswith("1.csv") and args.experimental_setting in x]
all_files = [x for x in os.listdir(pred_dir) if x.endswith("1.json")]
all_files

['qwen_3_8B_variable_DD_app_name_only_1.json',
 'gemma_4_31B_it_variable_DD_app_name_only_1.json',
 'gpt_oss_120b_variable_DD_app_name_only_1.json',
 'gpt_oss_20b_variable_DD_app_name_only_1.json',
 'llama3_3_70b_variable_DD_app_name_only_1.json',
 'qwen_3_30b_a3b_thinking_variable_DD_app_name_only_1.json',
 'claude_sonnet_4_5_variable_DD_app_name_only_1.json',
 'llama4_17b_16e_variable_DD_app_name_only_1.json']

In [37]:
all_settings = list(set([x.split("_variable_DD")[1].replace(".json", "") for x in all_files]))
all_settings

['_app_name_only_1']

In [ ]:
def get_last_json(output: str):
    """
    Extracts and returns the last valid JSON object or list from a given text.
    
    Parameters:
        output (str): The text containing JSON snippets.
    
    Returns:
        dict | list | None: The last parsed JSON object/list, or None if none found.
    """
    # Regex to capture JSON arrays or objects
    json_pattern = r'(\{.*?\}|\[.*?\])'
    last_valid = "Invalid Output!"

    try:
        matches = re.findall(json_pattern, output, flags=re.DOTALL)
        
    except TypeError:
            return "Invalid Output!"
        
    
    for match in matches:
        try:
            parsed = json.loads(match)
            last_valid = parsed
        except json.JSONDecodeError:
            continue  # Skip invalid matches

        
    
    return last_valid
    
def process_predictions(prediction):
    """Accept a generated string and process it according to the regex specified. 
    In this case, by replacing special characters (very common in the context of short description generation) with a blank char, then split according to camel casing such that the  and convert to lower case"""

    if prediction is None:
        prediction="None"

    cleaned_prediction = re.sub(r'[^a-zA-Z0-9]', '', prediction)

    result = re.findall(r'[A-Z]+(?![a-z])|[A-Z][a-z]*|[a-z]+', cleaned_prediction)

    return " ".join(result).lower()


def replace_with_spaced(strings):
    """
    Replace non-spaced strings with spaced versions if both exist and are equivalent ignoring spaces.
    """
    # Build mapping of key -> preferred version (spaced if available)
    preferred = {}
    for s in strings:
        key = s.replace(" ", "").lower()
        if key not in preferred or (" " in s and " " not in preferred[key]):
            preferred[key] = s

    # Rebuild list with replacements
    result = []
    for s in strings:
        key = s.replace(" ", "").lower()
        result.append(preferred[key])
    return result

In [40]:
def voted_output(predictions):
    """Accept the processed generation and returns its frequency"""
    pred_frequency = Counter(predictions)

    return pred_frequency
    

In [ ]:
data_df = pd.read_json(args.data_path)

In [44]:
if args.setting=="single_variable":
    selected_columns = ['variable_name', 'app_name', 'ParaID', 'ParaName', 'ProgID',
       'program_name', 'VarID', 'IsCopy', 'Father', 'Ancestor', 'Redefines',
       'Redefined', 'PIC', 'Type', 'IsField', 'NumOfChilds', 'VarOccurID',
       'VarOccOccurID', 'szValues', 'iLevel', 'ParaOccurID', 'ParaOccOccurID',
       'ParaStartRow', 'ParaEndRow', 'ParaStartCol', 'ParaEndCol',
       'VarStartRow', 'VarEndRow', 'VarStartCol', 'VarEndCol', 'ResourceID',
       'ResourceType', 'bRead', 'PATHSTR', 'PathID', 'db_name', 'rank',
       'business_concept', 'business_critical', 'reason_for_selection',
        'corrected_path', 'R1_id']

    selected_columns_pred = ['variable_name', 'app_name', 'R1_id', 'predictions_DD']
    join_columns_pred = ['variable_name', 'app_name', 'R1_id']

if args.setting=="batching":
    selected_columns = ['variable_name', 'app_name', 'program_name', 'ProgramID', 'VarID',
       'IsCopy', 'Father', 'Ancestor', 'Redefines', 'Redefined', 'PIC', 'Type',
       'IsField', 'NumOfChilds', 'VarOccurID', 'szValues', 'iLevel',
       'StatementType', 'StatementDescription', 'VarStartRow', 'VarEndRow',
       'PATHSTR', 'PathID', 'OccurID', 'db_name', 'rank', 'business_concept',
       'business_critical', 'reason_for_selection', 'program_name_annotation',
       'program_path', 'file_name', 'file_path', 'disagreement_score', 'R1_id',
       'corrected_path', 'batch_variables']
    
    selected_columns_pred = ['variable_name', 'app_name', 'R1_id', 'predictions_DD']
    join_columns_pred = ['variable_name', 'app_name', 'R1_id']

### Parse the Generated JSON

In [ ]:
data_dfs_parsed = {}

verbose = len(args.selected_models)


for current_exp_setting in all_settings:

    
    # file_name = "/{}_variable_DD{}.csv".format(args.selected_models[0], current_exp_setting)

    # current_file_path = pred_dir+ file_name

    data_df = pd.read_json(args.data_path)[selected_columns]
    
    for model_name in args.selected_models:
        # Example: gpt_oss_20b_variable_DD_icl_1.csv
    
        file_name = "/{}_variable_DD{}.json".format(model_name, current_exp_setting)
        
        current_file_path = pred_dir+ file_name
        current_df = pd.read_json(current_file_path)
        
        current_df = pd.merge(data_df, current_df[selected_columns_pred] , on=join_columns_pred)[selected_columns_pred]

        # Checks
        if sum(current_df['variable_name']==data_df['variable_name'])==300:
            print("Pass")
        
        else:
            print(f"Failed for {current_file_path}")
    
        
        predictions = list(current_df["predictions_DD"])
    
        data_df["predictions_{}".format(model_name)] = predictions
        
        parsed_jsons = []
        
        for i in range(len(current_df)):
            current_output = predictions[i]
            
            if "record_batching" in current_exp_setting:
                parsed_output = get_last_json(output=current_output)
                
                if parsed_output!="Invalid Output!":
                     parsed_output =  [parsed_output[0]]
                    

            else:
                parsed_output = get_last_json(output=current_output)
                
            parsed_jsons.append(parsed_output)

            
        if "record_batching" in current_exp_setting:
            for curr_column in selected_columns:
                curr_column_val = data_df[curr_column]
                data_df[curr_column] = [[x] for x in curr_column_val]
                
            data_df["parsed_output_{}".format(model_name)] = parsed_jsons

        else:
            data_df["parsed_output_{}".format(model_name)] = parsed_jsons


            

    # Voting Logic
    if args.include_voting:
        predictions_short_descriptions = []
        predictions_long_descriptions = []
        disagreement_score = []
        
        formatted_output = []
        formatted_output_masked = []
        
    
        for i, row in data_df.iterrows():
        
            current_predictions_sd = []
            current_predictions_ld = []

            current_formatted_output = {'variable_name': row['variable_name']}
            current_formatted_output_masked = {'variable_name': row['variable_name']}

            
            for model_idx, model_name in enumerate(args.selected_models):
                if verbose:
                    print(f"{model_idx+1}: {model_name}")
                    verbose -=1
                
                current_model_pred = row["parsed_output_{}".format(model_name)]


                # Include Short Descriptions
        
                try:
                    
                    # processed_model_output_sd = process_predictions(current_model_pred[0]['short description']) # Used in the initial phase for annotation selection
                    processed_model_output_sd = current_model_pred[0]['short description']
        
                except:
                    
                    try:
                        # processed_model_output_sd = process_predictions(current_model_pred['short description']) # Used in the initial phase for annotation selection
                        processed_model_output_sd = current_model_pred['short description']
                        
        
                    except:
                        processed_model_output_sd = "Invalid Output!"


                # Include Long Descriptions
        
                try:
                    processed_model_output_ld = current_model_pred[0]['long description']
        
                except:
                    try:
                        processed_model_output_ld = current_model_pred['long description']
        
                    except:
                        processed_model_output_ld = "Invalid Output!"

                        
                    
                    
                
                current_predictions_sd.append(processed_model_output_sd)

                # Format Long Description for Annotation
                current_predictions_ld.append(tuple(["{{"+processed_model_output_ld+"}}"]))
                

                current_formatted_output[model_name] = {
                    "short_description":processed_model_output_sd,
                    "long_description": processed_model_output_ld
                }

                current_formatted_output_masked[f"Model_{model_idx+1}"] = {
                    "short_description":processed_model_output_sd,
                    "long_description": processed_model_output_ld
                }
        
            
            
            # Output Formatting
            formatted_output.append(current_formatted_output)
            formatted_output_masked.append(current_formatted_output_masked)
            
            
            # Voting 
            processed_current_predictions_sd = replace_with_spaced(current_predictions_sd)
            voting_agg = dict(voted_output(processed_current_predictions_sd))
            predictions_short_descriptions.append(voting_agg)
            disagreement_score.append(len(voting_agg))
            predictions_long_descriptions.append(current_predictions_ld)

        
        data_df["predictions_voting_sd"] = predictions_short_descriptions
        data_df["disagreement_score"] = disagreement_score
        data_df["predictions_voting_ld"] = predictions_long_descriptions


        # Formatted Output For JSON
        data_df["formatted_output"] = formatted_output
        data_df["formatted_output_masked"] = formatted_output_masked

    data_dfs_parsed[current_exp_setting] = data_df

Pass
Pass
Pass
Pass
Pass
Pass
Pass
Pass
1: llama3_3_70b
2: gpt_oss_120b
3: gpt_oss_20b
4: qwen_3_8B
5: llama4_17b_16e
6: qwen_3_30b_a3b_thinking
7: gemma_4_31B_it
8: claude_sonnet_4_5


In [50]:
for current_exp_setting in all_settings:
    saving_file_path = args.saving_path+args.setting+"/complete_formatted_voting{}.json".format(current_exp_setting)
    print(saving_file_path)
    current_df_var = data_dfs_parsed[current_exp_setting]
    current_df_var.to_json(saving_file_path, orient="records", index=False)
    

../saved_data/processed_inference/single_variable/complete_formatted_voting_app_name_only_1.json
